# Evaluation Notebook — LangGraph RAG vs Baseline

Task 5: run ≥20 questions through the graph system and the simple baseline, then label accuracy and compare.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.pipeline import ingest_and_index
from src.graph.runner import run_rag
from src.graph.builder import build_graph, export_graph_mermaid
from src.generation.chains import run_baseline
from tests.evaluation.questions import SAMPLE_QUESTIONS
from tests.evaluation.metrics import run_eval_suite, summarize_comparison

print("ROOT =", ROOT)
print("Questions available:", len(SAMPLE_QUESTIONS))

ROOT = c:\Users\manickam.s\OneDrive - InTimeTec Visionsoft Pvt. Ltd.,\Documents\Multi-Agent_RAG_System
Questions available: 20


## 0. Index corpus (skip if already built)

In [ ]:
# ingest_and_index()

## 1. Graph visualization (Task 6)

In [2]:
graph = build_graph()
mmd = export_graph_mermaid(graph)
assets = ROOT / "assets"
assets.mkdir(exist_ok=True)
(assets / "graph.mmd").write_text(mmd, encoding="utf-8")
print(mmd)

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	query_analyser(query_analyser)
	query_rewriter(query_rewriter)
	retriever(retriever)
	relevance_grader(relevance_grader)
	answer_generator(answer_generator)
	hallucination_checker(hallucination_checker)
	__end__([<p>__end__</p>]):::last
	__start__ --> query_analyser;
	answer_generator --> hallucination_checker;
	hallucination_checker -. &nbsp;end&nbsp; .-> __end__;
	hallucination_checker -.-> answer_generator;
	query_analyser --> query_rewriter;
	query_rewriter --> retriever;
	relevance_grader -.-> answer_generator;
	relevance_grader -.-> retriever;
	retriever --> relevance_grader;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 2. Run evaluation (≥20 questions)

In [3]:
questions = SAMPLE_QUESTIONS  # replace with mentor questions when available
assert len(questions) >= 20

# Pause between questions to reduce Gemini free-tier 503 / rate-limit failures.
# Checkpoints are saved after every question so a crash does not lose progress.
TRACE_DIR = ROOT / "data" / "traces"
TRACE_DIR.mkdir(parents=True, exist_ok=True)

df_graph = run_eval_suite(
    questions,
    lambda q: run_rag(q),
    "langgraph",
    pause_seconds=4.0,
    checkpoint_path=TRACE_DIR / "eval_langgraph_checkpoint.csv",
)
df_base = run_eval_suite(
    questions,
    lambda q: run_baseline(q),
    "baseline",
    pause_seconds=3.0,
    checkpoint_path=TRACE_DIR / "eval_baseline_checkpoint.csv",
)
df = pd.concat([df_graph, df_base], ignore_index=True)
df.to_csv(TRACE_DIR / "eval_results.csv", index=False)
print("Saved", TRACE_DIR / "eval_results.csv")
print("Failures:", (df["error"].astype(str).str.len() > 0).sum())
df.head()

[langgraph] Resuming from checkpoint 'eval_langgraph_checkpoint.csv'. Already completed 8 / 20 questions.
[langgraph] (1/20) [SKIP - Already Completed] What is Retrieval-Augmented Generation?...
[langgraph] (2/20) [SKIP - Already Completed] Why do linear RAG pipelines fail on complex questions?...
[langgraph] (3/20) [SKIP - Already Completed] How does LangGraph help build stateful multi-agent RAG syste...
[langgraph] (4/20) [SKIP - Already Completed] What are the four query types used in research assistants?...
[langgraph] (5/20) [SKIP - Already Completed] Explain multi-query retrieval and why it improves recall....
[langgraph] (6/20) [SKIP - Already Completed] What does a relevance grader do in a RAG graph?...
[langgraph] (7/20) [SKIP - Already Completed] How does a hallucination checker improve answer faithfulness...
[langgraph] (8/20) [SKIP - Already Completed] What evaluation metrics are recommended for comparing RAG sy...
[langgraph] (9/20) Running: What is parent-document retriev

c:\Users\manickam.s\OneDrive - InTimeTec Visionsoft Pvt. Ltd.,\Documents\Multi-Agent_RAG_System\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.
c:\Users\manickam.s\OneDrive - InTimeTec Visionsoft Pvt. Ltd.,\Documents\Multi-Agent_RAG_System\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

c:\Users\manickam.s\OneDrive - InTimeTec Visionsoft Pvt. Ltd.,\Documents\Multi-Agent_RAG_System\src\retrieval\retriever.py:29: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='2cd3d867-17da-48dc-a5ce-761302ca28e6', metadata={'source': 'C:\\Users\\manickam.s\\OneDrive - InTimeTec Visionsoft Pvt. Ltd.,\\Documents\\Multi-Agent_RAG_System\\data\\raw\\sample_corpus.md', 'chunk_id': 3, 'file_path': 'C:\\Users\\manickam.s\\OneDrive - InTimeTec Visionsoft Pvt. Ltd.,\\Documents\\Multi-Agent_RAG_System\\data\\raw\\sample_corpus.md'}, page_content='## Multi-Query Retrieval\n\nMulti-query retrieval generates several rephrasings of the user question,\nretrieves documents for each rephrasing, then merges and deduplicates results.\nThis improves recall when wording mismatch would otherwise hide relevant text.\n\n## Relevance Grading and Hallucination Checking\n\nA relevance grader scores each retrieved chunk against the original question.\nIf average relevance falls below a t

  -> OK (133.5s) | retries: r=1, g=0
[langgraph] (10/20) Running: What is step-back prompting?...


...

c:\Users\manickam.s\OneDrive - InTimeTec Visionsoft Pvt. Ltd.,\Documents\Multi-Agent_RAG_System\src\retrieval\retriever.py:29: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='aa9ce36e-db29-4871-8333-2d5edc6249bb', metadata={'source': 'C:\\Users\\manickam.s\\OneDrive - InTimeTec Visionsoft Pvt. Ltd.,\\Documents\\Multi-Agent_RAG_System\\data\\raw\\sample_corpus.md', 'file_path': 'C:\\Users\\manickam.s\\OneDrive - InTimeTec Visionsoft Pvt. Ltd.,\\Documents\\Multi-Agent_RAG_System\\data\\raw\\sample_corpus.md', 'chunk_id': 1}, page_content='# Sample Research Corpus — AI Safety & RAG Systems\n\n## What is Retrieval-Augmented Generation?\n\nRetrieval-Augmented Generation (RAG) combines a retriever with a generator.\nThe retriever finds relevant passages from a document corpus. The generator\n(usually a large language model) produces an answer grounded in those passages.\nRAG reduces hallucinations compared to asking an LLM with no external context.\n\n## Limitations

  -> OK (4.8s) | retries: r=0, g=0
[baseline] (20/20) Running: How does multi-query retrieval merge results?...


c:\Users\manickam.s\OneDrive - InTimeTec Visionsoft Pvt. Ltd.,\Documents\Multi-Agent_RAG_System\src\retrieval\retriever.py:29: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='2cd3d867-17da-48dc-a5ce-761302ca28e6', metadata={'chunk_id': 3, 'source': 'C:\\Users\\manickam.s\\OneDrive - InTimeTec Visionsoft Pvt. Ltd.,\\Documents\\Multi-Agent_RAG_System\\data\\raw\\sample_corpus.md', 'file_path': 'C:\\Users\\manickam.s\\OneDrive - InTimeTec Visionsoft Pvt. Ltd.,\\Documents\\Multi-Agent_RAG_System\\data\\raw\\sample_corpus.md'}, page_content='## Multi-Query Retrieval\n\nMulti-query retrieval generates several rephrasings of the user question,\nretrieves documents for each rephrasing, then merges and deduplicates results.\nThis improves recall when wording mismatch would otherwise hide relevant text.\n\n## Relevance Grading and Hallucination Checking\n\nA relevance grader scores each retrieved chunk against the original question.\nIf average relevance falls below a t

  -> OK (3.5s) | retries: r=0, g=0
[baseline] Checkpoint updated at: c:\Users\manickam.s\OneDrive - InTimeTec Visionsoft Pvt. Ltd.,\Documents\Multi-Agent_RAG_System\data\traces\eval_baseline_checkpoint.csv
Saved c:\Users\manickam.s\OneDrive - InTimeTec Visionsoft Pvt. Ltd.,\Documents\Multi-Agent_RAG_System\data\traces\eval_results.csv
Failures: 2


,question,system,answer,accuracy,retry_triggered,retrieval_retries,generation_retries,hallucination_flagged,latency_seconds,error
0,What is Retrieval-Augmented Generation?,langgraph,"Based on the provided document, Retrieval-Augm...",NaN,False,0,0,False,31.845,NaN
1,Why do linear RAG pipelines fail on complex qu...,langgraph,"Based on the provided context, linear RAG pipe...",NaN,False,0,0,False,15.570,NaN
2,How does LangGraph help build stateful multi-a...,langgraph,"Based on the provided context, LangGraph helps...",NaN,False,0,0,False,18.337,NaN
3,What are the four query types used in research...,langgraph,"Based on the provided document, the four query...",NaN,False,0,0,False,15.917,NaN
4,Explain multi-query retrieval and why it impro...,langgraph,"Based on the provided documents, multi-query r...",NaN,False,0,0,False,16.184,NaN


## 3. Label accuracy manually

Set each row's `accuracy` to one of: `correct`, `partially_correct`, `incorrect`, `hallucinated`, `unable_to_answer`.

In [4]:
# Export for labeling (also already saved by the eval cell)
out = ROOT / "data" / "traces" / "eval_results.csv"
df.to_csv(out, index=False)
print("Wrote", out)

# After you fill the accuracy column in the CSV (or here), summarize:
# df = pd.read_csv(out)
# summarize_comparison(df)

Wrote c:\Users\manickam.s\OneDrive - InTimeTec Visionsoft Pvt. Ltd.,\Documents\Multi-Agent_RAG_System\data\traces\eval_results.csv


## 4. Multi-query vs plain similarity (Task 3, ≥15 questions)

In [5]:
subset = questions[:15]
df_mq = run_eval_suite(
    subset,
    lambda q: run_rag(q, use_multi_query=True),
    "multi_query",
    pause_seconds=4.0,
    checkpoint_path=TRACE_DIR / "eval_mq_checkpoint.csv",
)
df_plain = run_eval_suite(
    subset,
    lambda q: run_rag(q, use_multi_query=False),
    "plain_similarity",
    pause_seconds=4.0,
    checkpoint_path=TRACE_DIR / "eval_plain_checkpoint.csv",
)
df_enh = pd.concat([df_mq, df_plain], ignore_index=True)
df_enh.to_csv(TRACE_DIR / "eval_enhancement.csv", index=False)
df_enh[["question", "system", "latency_seconds", "retrieval_retries", "error"]].head(10)

[multi_query] (1/15) Running: What is Retrieval-Augmented Generation?...


c:\Users\manickam.s\OneDrive - InTimeTec Visionsoft Pvt. Ltd.,\Documents\Multi-Agent_RAG_System\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\manickam.s\OneDrive - InTimeTec Visionsoft Pvt. Ltd.,\Documents\Multi-Agent_RAG_System\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\manickam.s\OneDrive - InTimeTec Visionsoft Pvt. Ltd.,\Documents\Multi-Agent_RAG_System\src\retrieval\retriever.py:29: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='aa9ce36e-db29-4871-8333-2d5edc6249bb', metadata={'source': 'C:\\Users\\manickam.s\\OneDrive - InTimeTec Visionsoft Pvt. Ltd.,\\Document

  -> OK (11.9s) | retries: r=0, g=0
[multi_query] (2/15) Running: Why do linear RAG pipelines fail on complex questions?...


c:\Users\manickam.s\OneDrive - InTimeTec Visionsoft Pvt. Ltd.,\Documents\Multi-Agent_RAG_System\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\manickam.s\OneDrive - InTimeTec Visionsoft Pvt. Ltd.,\Documents\Multi-Agent_RAG_System\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\manickam.s\OneDrive - InTimeTec Visionsoft Pvt. Ltd.,\Documents\Multi-Agent_RAG_System\src\retrieval\retriever.py:29: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='8cde21c3-f016-4ba1-b168-a31e56a6cc91', metadata={'chunk_id': 4, 'source': 'C:\\Users\\manickam.s\\OneDrive - InTimeTec Visionsoft Pvt. 

  -> OK (33.3s) | retries: r=0, g=0
[multi_query] (3/15) Running: How does LangGraph help build stateful multi-agent RAG systems?...


...

c:\Users\manickam.s\OneDrive - InTimeTec Visionsoft Pvt. Ltd.,\Documents\Multi-Agent_RAG_System\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\manickam.s\OneDrive - InTimeTec Visionsoft Pvt. Ltd.,\Documents\Multi-Agent_RAG_System\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\manickam.s\OneDrive - InTimeTec Visionsoft Pvt. Ltd.,\Documents\Multi-Agent_RAG_System\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\U

  -> OK (24.5s) | retries: r=0, g=0
[plain_similarity] (15/15) Running: Why is typed shared state important in multi-agent graphs?...


c:\Users\manickam.s\OneDrive - InTimeTec Visionsoft Pvt. Ltd.,\Documents\Multi-Agent_RAG_System\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


  -> FAIL (100.2s) | [WinError 10053] An established connection was aborted by the software in your host machine
[plain_similarity] Checkpoint updated at: c:\Users\manickam.s\OneDrive - InTimeTec Visionsoft Pvt. Ltd.,\Documents\Multi-Agent_RAG_System\data\traces\eval_plain_checkpoint.csv


,question,system,latency_seconds,retrieval_retries,error
0,What is Retrieval-Augmented Generation?,multi_query,11.885,0,
1,Why do linear RAG pipelines fail on complex qu...,multi_query,33.325,0,
2,How does LangGraph help build stateful multi-a...,multi_query,8.608,0,
3,What are the four query types used in research...,multi_query,20.197,0,
4,Explain multi-query retrieval and why it impro...,multi_query,11.202,0,
5,What does a relevance grader do in a RAG graph?,multi_query,18.808,0,
6,How does a hallucination checker improve answe...,multi_query,36.541,0,
7,What evaluation metrics are recommended for co...,multi_query,35.781,0,
8,What is parent-document retrieval?,multi_query,13.270,1,
9,What is step-back prompting?,multi_query,10.912,1,


## 5. Save one complex state trace (Task 4)

In [ ]:
# At the top of cell 5, add a fresh import guard:
import importlib, src.graph.runner
importlib.reload(src.graph.runner)

complex_q = "Compare factual and multi-hop questions and explain how recovery loops help RAG."
trace = run_rag(complex_q, save_trace=True, trace_name="complex_query_trace")
print(trace["final_answer"][:500])
print("Trace:", trace.get("trace_path"))


c:\Users\manickam.s\OneDrive - InTimeTec Visionsoft Pvt. Ltd.,\Documents\Multi-Agent_RAG_System\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


ConnectError: [Errno 11001] getaddrinfo failed